# P&ID Stage 2 — encoder one-shot toàn cục trên Colab (GPU)

Train **một** encoder ResNet-34 + ArcFace trên crop DigitizePID (`fgc.global_encoder=true`),
rồi dựng prototype từ template ISO 10628-2. Thứ tự (docs/PLAN_STAGE2_ONESHOT.md, Phase B;
hướng dẫn từng bước: docs/COLAB_RUNBOOK.md §5c):

| Bước | Mục | Ra file trên Drive (`runs/encoders/`) |
|---|---|---|
| R1b — encoder đủ 32 class (lõi luận văn) — **chạy TRƯỚC** | 6 | `__global__.pt` |
| R1a — encoder **leave-out** (protocol P1) | 5 | `__global_lo__.pt` |
| R2 — prototype ISO + separation | 7 | `iso_prototypes.pt`, `iso_prototypes_lo.pt` |

Chạy theo thứ tự mục: 1 → 2 → 3 → 4 → **6** → 5 → 7 (mục 6 nằm sau mục 5 trong file).
R1b đã xong (2026-11-12): phiên R1a chạy 1 → 2 → 3 → 4 → 5 → 7.

**Trước khi chạy**: nhánh đã **push** lên GitHub (Colab clone code từ đó); danh sách H đã chốt trong
`configs/config.yaml` (`oneshot.holdout`, 2026-11-11). Mục 4 dừng nếu danh sách trống.
**Runtime**: GPU (T4 đủ). Crop + augment chạy trên CPU (`LOADER` ở mục 4).

## 1. GPU + mount Drive

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
import os
# Thư mục dự án trên Drive phải chứa data/canonical/. Notebook Stage 1 dùng MyDrive/pid, bản
# Stage 2 cũ dùng MyDrive/PID_Recognition -> thử cả hai; sửa DRIVE_ROOT nếu ở chỗ khác.
DRIVE_ROOT = next((d for d in ('/content/drive/MyDrive/pid', '/content/drive/MyDrive/PID_Recognition')
                   if os.path.isdir(f'{d}/data/canonical')), None)
assert DRIVE_ROOT, 'Không thấy <DRIVE_ROOT>/data/canonical — sửa DRIVE_ROOT bằng tay'
DATA_ROOT = f'{DRIVE_ROOT}/data'
RUNS_DIR  = f'{DRIVE_ROOT}/runs'
ENC_DIR   = f'{RUNS_DIR}/encoders'      # encoder + prototype đã đặt tên rõ ràng
os.makedirs(ENC_DIR, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)
print('canonical images:', len(os.listdir(f'{DATA_ROOT}/canonical/images')))

## 2. Code + deps

In [ ]:
REPO_URL = 'https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git'   # <-- sửa
BRANCH   = 'feat/prepare-for-colab-stage-2'
%cd /content
![ -d pid ] || git clone -b $BRANCH $REPO_URL pid
%cd /content/pid
!git fetch -q && git checkout -q $BRANCH && git pull -q && git log --oneline -1
!pip install -q uv
# Colab đã có torch CUDA -> không cài torch. run_pipeline.py import mọi stage (cả detector),
# nên cần đủ ultralytics/sahi/ensemble-boxes dù Stage 2 không dùng tới.
!uv pip install --system -q ultralytics sahi ensemble-boxes albumentations timm hydra-core omegaconf \
    pycocotools opencv-python-headless pillow pandas scikit-learn matplotlib tqdm pyyaml
import sys
os.environ['PYTHONPATH'] = '/content/pid/src'
sys.path.insert(0, '/content/pid/src')
REPO_DATA = '/content/pid/data/raw'     # template ISO + vocab.csv nằm trong git, không ở Drive

## 3. Crop toàn cục — đủ 32 class

**Bẫy im lặng**: `derived/crops_fgc/` và `crops_fgc.zip` cũ trên Drive chỉ có **28 class** (cắt theo
`fgc_group`). Notebook này KHÔNG BAO GIỜ đọc chúng: crop toàn cục được cắt ở local rồi lưu thành
`crops_fgc_ctx.zip` (tên khác). Cắt vào thư mục cũ cũng không được — tên file đánh số khác
nhau giữa hai chế độ nên sẽ ra crop trùng.

**Định dạng crop mới (2026-11-11, `utils/crop_box.py`)**: mỗi PNG là vùng ngữ cảnh quanh ký hiệu, kèm
`<lớp>/_boxes.csv` (vị trí box). Dataset cắt ra view box + 10% (giống lúc suy luận), hoặc view lệch
ngẫu nhiên khi train (`box_jitter`). Zip cũ `crops_fgc_global.zip` (không có `_boxes.csv`) **không dùng**
nữa — cell dưới kiểm `_boxes.csv` ở mọi lớp.

Phiên đầu: copy ảnh canonical (~1.2 GB) về local, cắt, zip lên Drive. Phiên sau: chỉ giải nén.

In [ ]:
import glob, shutil, time
from utils.classmap import load_classmap
LOCAL_DATA_ROOT = '/content/data'
LOCAL_CROPS = f'{LOCAL_DATA_ROOT}/derived/crops_fgc'
GLOBAL_ZIP  = f'{DATA_ROOT}/derived/crops_fgc_ctx.zip'
t0 = time.time()
os.makedirs(f'{LOCAL_DATA_ROOT}/canonical', exist_ok=True)
os.makedirs(f'{LOCAL_DATA_ROOT}/derived', exist_ok=True)
for f in ('classes.yaml', 'manifest.csv', 'annotations.coco.json', 'digitizepid_splits.json'):
    if os.path.exists(f'{DATA_ROOT}/canonical/{f}'):
        shutil.copy2(f'{DATA_ROOT}/canonical/{f}', f'{LOCAL_DATA_ROOT}/canonical/{f}')

if os.path.exists(GLOBAL_ZIP):
    !unzip -q -o "$GLOBAL_ZIP" -d "$LOCAL_DATA_ROOT/derived/"
else:
    !rsync -a "$DATA_ROOT/canonical/images/" "$LOCAL_DATA_ROOT/canonical/images/"
    for step, f in (('classes', 'classes.yaml'), ('manifest', 'manifest.csv')):
        if not os.path.exists(f'{LOCAL_DATA_ROOT}/canonical/{f}'):
            !cd /content/pid && python src/run_pipeline.py prep --step $step data_root=$LOCAL_DATA_ROOT
    !cd /content/pid && python src/run_pipeline.py prep --step crops data_root=$LOCAL_DATA_ROOT fgc.global_encoder=true
    !cd "$LOCAL_DATA_ROOT/derived" && zip -q -r crops_fgc_ctx.zip crops_fgc && cp crops_fgc_ctx.zip "$GLOBAL_ZIP"
    print('Đã lưu', GLOBAL_ZIP)
print(f'Crops local sau {time.time()-t0:.0f}s')

# Phải đủ ĐÚNG mọi fine class của classes.yaml ở cả hai split (32 với DigitizePID).
CM = load_classmap(f'{LOCAL_DATA_ROOT}/canonical/classes.yaml')
for split in ('train', 'val'):
    present = set(os.listdir(f'{LOCAL_CROPS}/{split}'))
    n = len(glob.glob(f'{LOCAL_CROPS}/{split}/*/*.png'))
    print(f'  {split}: {n} crops, {len(present)} fine classes')
    missing = sorted(set(CM.fine_classes) - present)
    assert not missing, f'{split} thiếu class {missing} — zip cũ 28 class? xoá {GLOBAL_ZIP} rồi chạy lại'
    assert len(present) == len(CM.fine_classes) == 32, sorted(present)
    no_boxes = sorted(c for c in present if not os.path.exists(f'{LOCAL_CROPS}/{split}/{c}/_boxes.csv'))
    assert not no_boxes, f'{split}: thiếu _boxes.csv ở {no_boxes} — crop định dạng cũ? xoá {GLOBAL_ZIP} rồi chạy lại'

## 4. Danh sách giữ ra H (từ config, KHÔNG gõ tay)

Đọc `oneshot.holdout` trong `configs/config.yaml` (chốt 2026-11-11). R1a train với
`fgc.exclude_classes = h_in + h_out + h_extra` (`h_extra`: VARIANT cùng ký hiệu ISO với một lớp
`h_in`, bỏ khỏi train để không rò rỉ, không thi). Tên sai -> lỗi ngay tại đây, trước khi tốn GPU.

Cell này cũng đặt `LOADER` (2 worker nạp dữ liệu song song) và `AUG` (augment cho **cả R1a và R1b**, kể cả khi resume): suy giảm kiểu scan +
lệch box (PLAN, quyết định 2026-11-11). H phải chốt trước **cả hai** lượt train, nên R1b cũng cần
cell này. **R1b chạy trước R1a** (mục 6 trước mục 5).

In [ ]:
from omegaconf import OmegaConf
from utils.classmap import GLOBAL_GROUP, training_fine_classes
from utils.oneshot import holdout_classes
CFG = OmegaConf.load('/content/pid/configs/config.yaml')
H_IN, H_OUT, H_EXTRA = holdout_classes(CFG)             # trùng lặp giữa các list -> ValueError
H = H_IN + H_OUT                                         # lớp được thi ở P1 (H_EXTRA: bỏ khỏi train, KHÔNG thi)
assert H, 'oneshot.holdout trống: làm B5 (chọn H_in/H_out), sửa configs/config.yaml, push, rồi chạy lại cell 2'
HELD = H + H_EXTRA                                       # mọi lớp R1a không được thấy
KEPT = training_fine_classes(CM, GLOBAL_GROUP, HELD)    # tên không có trong classes.yaml -> ValueError
EXCLUDE = '[' + ','.join(HELD) + ']'
print(f'H_in    ({len(H_IN)}):', H_IN)
print(f'H_out   ({len(H_OUT)}):', H_OUT)
print(f'h_extra ({len(H_EXTRA)}):', H_EXTRA)
print(f'R1a train trên {len(KEPT)}/{len(CM.fine_classes)} class')

# Augment cho R1a VÀ R1b, kể cả resume (PLAN 2026-11-11). '++' thêm hoặc ghi đè key.
AUG = '++fgc.aug.degrade.p=0.5 ++fgc.aug.box_jitter.p=0.5'
# Augment chạy trên CPU; 2 worker = số nhân CPU của Colab thường. 0 = như cũ (chậm hơn).
LOADER = '++fgc.workers=2'
print('AUG:', AUG, '| LOADER:', LOADER)

# Quy tắc checkpoint (PLAN, chốt 2026-11-13): val_acc bão hoà từ epoch 1 nên "best val_acc" chọn
# epoch ngẫu nhiên. R1a và R1b so ở CÙNG epoch CMP_EPOCH. fgc_train ghi __global__.ep<N>.pt ở mỗi
# epoch trong save_epochs, và __global__.last.pt mỗi epoch (resume tiếp từ đó).
CMP_EPOCH = 3
EPOCHS = 5
SAVE = '++fgc.save_epochs=[3,5]'   # trong lệnh: "$SAVE" có ngoặc kép (dấu [ ] là glob của shell)
print('EPOCHS:', EPOCHS, '| SAVE:', SAVE, '| so ở epoch', CMP_EPOCH)

**4b. Hàm `finalize`** — chạy một lần mỗi phiên, trước 5b / 6b. Lấy checkpoint epoch `CMP_EPOCH`
của run, kiểm encoder không có cột nào của các lớp bị giữ ra, rồi copy vào `encoders/`.

In [ ]:
import torch
def finalize(run, out_name, expect_excluded, epoch=CMP_EPOCH):
    # Mọi phiên của run (kể cả phiên resume), .ep<N>.pt trước. Run trước 2026-11-13 không có
    # __global__.ep<N>.pt: khi đó dùng __global__.pt nếu nó đúng là epoch đó.
    newest = lambda pat: sorted(glob.glob(f'{RUNS_DIR}/{run}/*/fgc_checkpoints/{pat}'), key=os.path.getmtime, reverse=True)
    for ck in newest(f'__global__.ep{epoch}.pt') + newest('__global__.pt'):
        st = torch.load(ck, map_location='cpu', weights_only=False)
        if st['epoch'] + 1 == epoch:
            break
    else:
        raise FileNotFoundError(f'{RUNS_DIR}/{run}: không có checkpoint epoch {epoch} — train chưa tới epoch {epoch}?')
    fine = list(st['fine_classes'])
    assert not set(expect_excluded) & set(fine), f'RÒ RỈ: {sorted(set(expect_excluded) & set(fine))} có trong encoder'
    assert len(fine) == len(CM.fine_classes) - len(expect_excluded), (len(fine), len(expect_excluded))
    assert sorted(st.get('excluded_classes', [])) == sorted(expect_excluded), st.get('excluded_classes')
    assert st['model']['head.weight'].shape[0] == len(fine)
    shutil.copy2(ck, f'{ENC_DIR}/{out_name}')
    print(f'{ck}\n  -> {ENC_DIR}/{out_name}: {len(fine)} class, epoch {st["epoch"]+1}, val_acc {st["val_acc"]:.4f}')

## 5. R1a — encoder leave-out → `__global_lo__.pt`

Train `EPOCHS` epoch (mục 4). Checkpoint ghi thẳng vào Drive, `runs/r1a/<ts>/fgc_checkpoints/`:
`__global__.ep3.pt`, `__global__.ep5.pt` (epoch cố định), `__global__.last.pt` (mỗi epoch, để resume),
`__global__.pt` (best val_acc, KHÔNG dùng để so). Cell 5b lấy epoch `CMP_EPOCH` → `encoders/__global_lo__.pt`.

In [ ]:
!cd /content/pid && python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/r1a \
    fgc.global_encoder=true fgc.metric_learning=true fgc.epochs=$EPOCHS fgc.batch=128 $AUG $LOADER "$SAVE" \
    "fgc.exclude_classes=$EXCLUDE"

**5a. Resume R1a** (nếu Colab ngắt) — tự lấy run R1a mới nhất; CÙNG danh sách H (code chặn nếu khác).

In [ ]:
# Phiên mới nhất có checkpoint; fgc_train tự lấy __global__.last.pt trong đó (cũ: __global__.pt).
R1A_LAST = sorted(glob.glob(f'{RUNS_DIR}/r1a/*/fgc_checkpoints/__global__*.pt'), key=os.path.getmtime)[-1]
R1A_RESUME = os.path.dirname(R1A_LAST)
print('resume từ', R1A_RESUME)
!cd /content/pid && python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/r1a \
    fgc.global_encoder=true fgc.metric_learning=true fgc.epochs=$EPOCHS fgc.batch=128 $AUG $LOADER "$SAVE" \
    "fgc.exclude_classes=$EXCLUDE" fgc.resume_from=$R1A_RESUME

**5b. Kiểm + đặt tên** — encoder KHÔNG được có cột nào của H hay `h_extra`. Cần cell 4b.

In [ ]:
finalize('r1a', '__global_lo__.pt', HELD)  # h_in + h_out + h_extra, epoch CMP_EPOCH

## 6. R1b — encoder đủ 32 class → `__global__.pt`

Lõi luận văn (protocol C), P2, P4 và end-to-end. **Chạy trước R1a** (PLAN 2026-11-09). `fgc.exclude_classes=[]` ghi rõ để không thừa hưởng nhầm.

**Đã chạy 2026-11-12** (Colab ngắt sau epoch 3; `encoders/__global__.pt` = epoch 3 = `CMP_EPOCH`).
Không chạy lại mục 6 trừ khi đổi cách train (vd. chia B, nhãn theo nghĩa ISO).

In [ ]:
!cd /content/pid && python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/r1b \
    fgc.global_encoder=true fgc.metric_learning=true fgc.epochs=$EPOCHS fgc.batch=128 $AUG $LOADER "$SAVE" \
    "fgc.exclude_classes=[]"

**6a. Resume R1b** (nếu Colab ngắt).

In [ ]:
# Phiên mới nhất có checkpoint; fgc_train tự lấy __global__.last.pt trong đó (cũ: __global__.pt).
R1B_LAST = sorted(glob.glob(f'{RUNS_DIR}/r1b/*/fgc_checkpoints/__global__*.pt'), key=os.path.getmtime)[-1]
R1B_RESUME = os.path.dirname(R1B_LAST)
print('resume từ', R1B_RESUME)
!cd /content/pid && python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/r1b \
    fgc.global_encoder=true fgc.metric_learning=true fgc.epochs=$EPOCHS fgc.batch=128 $AUG $LOADER "$SAVE" \
    "fgc.exclude_classes=[]" fgc.resume_from=$R1B_RESUME

**6b. Kiểm + đặt tên.** Cần cell 4b.

In [ ]:
finalize('r1b', '__global__.pt', [])  # epoch CMP_EPOCH

## 7. R2 — prototype ISO + separation (GĐ2)

Mỗi encoder có sẵn trong `encoders/` -> một prototype directory. Đọc dòng
`prototype separation: mean off-diagonal cos=…, max=… (closest pair: A vs B)` và ghi số vào HANDOFF
(bảng B4 trong plan: max < ~0.8 tốt; mean ≥ ~0.6 = encoder không transfer -> B9 prototype có stub,
rồi BACKUP_PLAN; GĐ3 đã bỏ).

In [ ]:
for enc, out in (('__global_lo__.pt', 'iso_prototypes_lo.pt'), ('__global__.pt', 'iso_prototypes.pt')):
    if not os.path.exists(f'{ENC_DIR}/{enc}'):
        print('bỏ qua (chưa có)', enc); continue
    print(f'===== {enc} -> {out}')
    !cd /content/pid && python src/run_pipeline.py prototypes run_dir=$RUNS_DIR/r2 \
        oneshot.encoder_ckpt=$ENC_DIR/$enc oneshot.prototypes=$ENC_DIR/$out \
        oneshot.symbols_dir=$REPO_DATA/iso_symbols_clean oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv \
        2>&1 | grep -v "^Warning"

## 8. Eval closed-set (protocol C) — `__global__.pt` trên crop val, nửa `test`

Chạy sau mục 6b. `eval.mode=fgc_only` chấm head 32 lớp của R1b trên crop GT, **không cần detector**:
top-1/top-3, macro-F1, P/R/F1 từng lớp, confusion 32×32 (`confusion___global__.png`), tỉ lệ lỗi
nhảy sang họ coarse khác. Ra `runs/eval_c/<ts>/metrics.json` (PLAN B6c).

End-to-end (mAP 32 lớp fine, cạnh YOLO một-stage) cần thêm Stage 1: `detect` → `fgc`
(`fgc.global_encoder=true fgc.ckpt_dir=$ENC_DIR`) → `evaluate eval.mode=det_plus_fgc`; YOLO 32 lớp
chấm bằng `eval.mode=det_only detector.label_level=fine`. `++eval.image_subset=test`: chỉ 50 ảnh `test` của val (`configs/splits/digitizepid_val_test.json`), nửa còn lại `val_sel`
dành cho chọn τ (§9). Eval one-shot P1/P2: §9; P4 (OPEN100-ISO) chưa có.

In [ ]:
!cd /content/pid && python src/run_pipeline.py evaluate \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/eval_c eval.mode=fgc_only eval.split=val \
    ++eval.image_subset=test fgc.global_encoder=true fgc.metric_learning=true fgc.ckpt_dir=$ENC_DIR
import json
M_PATH = sorted(glob.glob(f'{RUNS_DIR}/eval_c/*/metrics.json'), key=os.path.getmtime)[-1]
m = json.load(open(M_PATH))['fgc']
print(M_PATH)
print(f"{m['n_samples']} crop, {m['n_classes']} lớp: top1 {m['top1']:.4f}  top3 {m['top3']:.4f}  "
      f"macro-F1 {m['macro_f1']:.4f}  nhảy họ {m['cross_family_error_rate']:.4f}")
for name, r in sorted(m['per_class'].items(), key=lambda kv: kv[1]['f1'])[:5]:  # 5 lớp yếu nhất
    print(f"  {name:10s} P {r['precision']:.3f}  R {r['recall']:.3f}  F1 {r['f1']:.3f}  n={r['support']}")

## 9. Eval one-shot mức crop (B6b-1) — chọn τ, P1, P2

Cần mục 7 (prototype) và mục 4 (`oneshot.holdout`). `eval.mode=oneshot` nhúng mỗi crop GT (view box + 10%) bằng
`oneshot.query_views`, gộp bằng `oneshot.aggregate=mean`, so với **mọi** prototype (không route theo coarse), không gate.
Mỗi lượt ghi `runs/eval_*/<ts>/metrics.json` (khóa `oneshot`) + `oneshot_<protocol>_crops.csv` (top-3 tên + cosine từng
crop, phân tích lại không cần GPU) + `oneshot_<protocol>_sweep.png`.

Thứ tự bắt buộc (PLAN B8): **9a chọn τ trên `val_sel`** trước, rồi 9b/9c mới chạm `test`. Hai cell 9a chỉ nhìn lớp
nhóm S mà encoder đã train; **không** ghi τ vào config. 9b/9c tự đọc τ của 9a và báo "Other" tại cả ba τ (`rule_a`, `rule_b`,
`rule_b_no_twins`) từ CSV; `other_at_tau` trong metrics.json là tại `oneshot.other_min_cosine` (placeholder 0.35), bỏ qua.

In [ ]:
# τ ở 9b/9c đọc lại từ metrics.json của cell này (không cần biến sống qua phiên).
import csv as _csv
import numpy as np
from utils.oneshot_metrics import other_prf
def chosen_taus(tag):
    o = json.load(open(sorted(glob.glob(f'{RUNS_DIR}/eval_tau_{tag}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
    return {r: o[r]['tau'] for r in ('rule_a', 'rule_b', 'rule_b_no_twins')}
def at_chosen_taus(o, tag, known_groups, unknown_groups):
    # P/R/F1 của "Other" + tỉ lệ crop biết được nhận VÀ gọi đúng, tại τ của 9a (PLAN B8: báo cả hai quy tắc).
    rows = list(_csv.DictReader(open(o['crops_csv'])))
    known = [r for r in rows if r['group'] in known_groups]
    kc = [float(r['best_cos']) for r in known]
    uc = [float(r['best_cos']) for r in rows if r['group'] in unknown_groups]
    for rule, t in chosen_taus(tag).items():
        ccr = np.mean([float(r['best_cos']) >= t and r['top1'] == r['expected'] for r in known])
        prf = other_prf(kc, uc, t) or {}
        print(f"  {rule:16s} τ={t:.4f}  biết: nhận+đúng {ccr:.4f}  |  Other: P {prf.get('precision')}  R {prf.get('recall')}  F1 {prf.get('f1')}")

# 9a. Chọn τ (quy tắc A: τ@TPR95; quy tắc B: bỏ prototype của chính lớp, có/không loại cặp ★ cos > 0.95), val_sel
import json
for tag, enc, protos in (('lo', '__global_lo__.pt', 'iso_prototypes_lo.pt'), ('full', '__global__.pt', 'iso_prototypes.pt')):
    if not os.path.exists(f'{ENC_DIR}/{enc}'):
        print('bỏ qua (chưa có)', enc); continue
    print(f'===== tau {enc}')
    !cd /content/pid && python src/run_pipeline.py evaluate \
        data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/eval_tau_$tag eval.mode=oneshot ++eval.oneshot.protocol=tau \
        ++eval.image_subset=val_sel oneshot.encoder_ckpt=$ENC_DIR/$enc oneshot.prototypes=$ENC_DIR/$protos \
        oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv oneshot.aggregate=mean 2>&1 | grep -v "^Warning"
    o = json.load(open(sorted(glob.glob(f'{RUNS_DIR}/eval_tau_{tag}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
    print(f"{enc}: {o['rule_a']['n_crops']} crop, {o['rule_a']['n_classes']} lớp {o['classes']}")
    print(f"  A  tau={o['rule_a']['tau']:.4f}   B  tau={o['rule_b']['tau']:.4f} (J={o['rule_b']['youden_j']:.3f})   "
          f"B bỏ cặp sinh đôi  tau={o['rule_b_no_twins']['tau']:.4f}")
    print('  cặp ★ cos > 0.95 bị loại:', [(p['class'], p['twin'], round(p['cos'], 3)) for p in o['rule_b_no_twins']['twin_pairs']])

**9b. P1** — encoder leave-out `__global_lo__.pt` + `iso_prototypes_lo.pt`, crop `h_in` / `h_out` từ cả train và val
(encoder chưa train trên chúng; không dùng `image_subset`). Guard rò rỉ: raise nếu lớp thi nằm trong `fine_classes`.
Số chính: AUROC + OSCR (không phụ thuộc τ); top-1 / top-3 macro `h_in`; P/R/F1 của "Other" tại `oneshot.other_min_cosine`.

In [ ]:
!cd /content/pid && python src/run_pipeline.py evaluate \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/eval_p1 eval.mode=oneshot ++eval.oneshot.protocol=p1 \
    oneshot.encoder_ckpt=$ENC_DIR/__global_lo__.pt oneshot.prototypes=$ENC_DIR/iso_prototypes_lo.pt \
    oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv oneshot.aggregate=mean 2>&1 | grep -v "^Warning"
M_P1 = sorted(glob.glob(f'{RUNS_DIR}/eval_p1/*/metrics.json'), key=os.path.getmtime)[-1]
o = json.load(open(M_P1))['oneshot']
print(M_P1)
print(f"h_in macro top1 {o['h_in']['macro_top1']:.4f}  top3 {o['h_in']['macro_top3']:.4f}  "
      f"AUROC {o['open_set']['auroc']:.4f}  OSCR {o['open_set']['oscr_area']:.4f}  "
      f"({o['open_set']['n_known']} known / {o['open_set']['n_unknown']} unknown crop)")
print('Tại τ chọn ở 9a (encoder lo):'); at_chosen_taus(o, 'lo', {'H_chon'}, {'OOV_heldout'})
for name, r in o['per_class'].items():
    print(f"  {name:10s} {r['group']:12s} n={r['n']:5d} top1 {r['top1']}  top3 {r['top3']}  hay đoán {r['most_predicted']}")

**9c. P2** — encoder đủ 32 lớp `__global__.pt` + `iso_prototypes.pt`, crop `val` nửa `test`: lớp S (đã thấy) và V
(biến thể) là "biết", `OOV_seen` là "Other đã thấy" (báo riêng, không phải lớp lạ thật: encoder đã train trên chúng).
Gắn nhãn "seen-class" khi báo cáo. Chạy **một lần**, sau khi τ ở 9a đã chốt.

In [ ]:
!cd /content/pid && python src/run_pipeline.py evaluate \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/eval_p2 eval.mode=oneshot ++eval.oneshot.protocol=p2 \
    ++eval.image_subset=test oneshot.encoder_ckpt=$ENC_DIR/__global__.pt oneshot.prototypes=$ENC_DIR/iso_prototypes.pt \
    oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv oneshot.aggregate=mean 2>&1 | grep -v "^Warning"
M_P2 = sorted(glob.glob(f'{RUNS_DIR}/eval_p2/*/metrics.json'), key=os.path.getmtime)[-1]
o = json.load(open(M_P2))['oneshot']
print(M_P2)
for g in ('S', 'V', 'V_of_S', 'OOV_seen'):
    b = o['groups'][g]
    print(f"  {g:9s} {b['n_classes']:2d} lớp {b['n_crops']:5d} crop  macro top1 {b['macro_top1']}  top3 {b['macro_top3']}")
print(f"AUROC {o['open_set']['auroc']}  OSCR {o['open_set']['oscr_area']}")
print('Tại τ chọn ở 9a (encoder full):'); at_chosen_taus(o, 'full', {'S', 'V'}, {'OOV_seen'})
print('UNSURE bị loại:', o['unsure_crops_excluded'])